In [ ]:
import os
import re
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

plt.style.use('bmh')

PYTHON_PATH = '/home/andreas/mambaforge/envs/symdel/bin/python'
WORKER_SCRIPTS = {
    'symscan': 'tmp/symscan_memory_worker.py',
    'symdel': 'tmp/symdel_memory_worker.py',
}
SEQ_FILE = 'tmp/emerson_sequences.txt'
MAX_DISTANCE = 1
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1

In [ ]:
import benchutils as bu

bu.describe_env()

In [ ]:
!mkdir -p tmp

In [ ]:
if not os.path.exists(SEQ_FILE):
    df = pd.concat([
        pd.read_csv(f'../data/emerson_rep{i}.zip', sep=',', compression='zip')
        for i in range(1, 15)
        ])
    seqs = df['cdr3'].drop_duplicates().sample(frac=1, random_state=0).reset_index(drop=True)
    len_seqs = len(seqs)
    with open(SEQ_FILE, 'w') as f:
        f.write('\n'.join(seqs) + '\n')
else:
    with open(SEQ_FILE) as f:
        len_seqs = len(f.readlines())


In [ ]:
len_seqs

In [ ]:
%%writefile tmp/symscan_memory_worker.py
# Standalone worker run as a fresh subprocess for each measurement, so that the peak RSS
# reported by `/usr/bin/time -v` reflects a single symscan.get_neighbors_within call in
# isolation rather than a cumulative/high-water mark across many calls in one process.
import sys

import symscan


def main():
    seq_file, n_sequence, max_distance = sys.argv[1:4]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    with open(seq_file) as f:
        seqs = [next(f).strip() for _ in range(n_sequence)]

    symscan.get_neighbors_within(seqs, max_distance=max_distance)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile tmp/symdel_memory_worker.py
# Standalone worker run as a fresh subprocess for each measurement, so that the peak RSS
# reported by `/usr/bin/time -v` reflects a single pyrepseq.symdel call in
# isolation rather than a cumulative/high-water mark across many calls in one process.
import sys

import pyrepseq


def main():
    seq_file, n_sequence, max_distance = sys.argv[1:4]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    with open(seq_file) as f:
        seqs = [next(f).strip() for _ in range(n_sequence)]


    pyrepseq.symdel(seqs, max_edits=max_distance)


if __name__ == '__main__':
    main()

In [ ]:
def measure_peak_memory_gb(algorithm, n_sequence, max_distance=MAX_DISTANCE):
    cmd = ['/usr/bin/time', '-v', PYTHON_PATH, WORKER_SCRIPTS[algorithm],
           SEQ_FILE, str(n_sequence), str(max_distance)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    match = re.search(r'Maximum resident set size \(kbytes\): (\d+)', result.stderr)
    if match is None:
        raise RuntimeError(result.stderr)
    return int(match.group(1)) / 1024**2

sizes = np.unique(np.geomspace(1e5, len_seqs, num=12, dtype=int))
sizes

In [ ]:
rows = []
for algorithm in WORKER_SCRIPTS:
    for n_sequence in sizes:
        for rep in range(N_REPS):
            memory_gb = measure_peak_memory_gb(algorithm, n_sequence)
            rows.append({'algorithm': algorithm, 'n_sequence': n_sequence,
                          'distance': MAX_DISTANCE, 'measure': 'memory_gb',
                          'memory_gb': memory_gb})
            print(algorithm, n_sequence, rep, memory_gb)
        if memory_gb > 64:
            print(f"Memory usage exceeded 64 GB for {algorithm} with {n_sequence} sequences. Stopping further measurements for this algorithm.")
            break

In [ ]:
mem_df = pd.DataFrame(rows)
mem_df.to_csv('../data/symscan_memory_benchmark.csv')